[Guide Index](README.md)


# 05b SPARQL Queries as RDF

`starlayer.sparql` can represent a SPARQL query as an RDF graph, using the `sqe:` ontology (specific to StarLayer). `sqe:` serializes text into a human-readable RDF syntax tree designed for storage, validation, viewing and editing. The RDF graph can be serialized back as plain text, or converted to a query object.

A corresponding SHACL shapes graph validates the `sqe:` graph and incorporates `shacl:ui` components that can be used in a SHACL editor.

See the [SPARQL guide](03-sparql.ipynb) for ordinary query operations. This guide is specifically about treating a *query itself* as RDF graph data you can encode, inspect, edit, and validate.


## What `sqe:` covers

`sqe:` represents SELECT, ASK and CONSTRUCT queries (`SELECT *`, `DISTINCT`/`REDUCED` and `CONSTRUCT WHERE` included) and every WHERE-clause element: triple patterns, property paths, `OPTIONAL`, `UNION`, `MINUS`, `GRAPH`, `FILTER` (with `EXISTS`/`NOT EXISTS`), `BIND`, `VALUES` and sub-selects. It also covers projected expressions, aggregates with `GROUP BY`/`HAVING`, `ORDER BY`/`LIMIT`/`OFFSET`, every expression builtin, and the SPARQL 1.2 additions: triple terms, reified triples and annotations, and base-direction strings.

It does not cover `DESCRIBE` or SPARQL Update, and a query that uses something it cannot represent raises an error instead of losing it. Reified-triple and annotation shorthand (`~ ?r`, `{| ... |}`) is stored in its expanded form, so it comes back from `sqe_tree_to_text()` expanded.

A query written with SPARQL 1.2 syntax runs as text: use `sqe_tree_to_text()` and run that. The query object from `sqe_tree_to_query()` is for ordinary SPARQL.


## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later steps reuse variables from earlier ones.


In [1]:
from starlayer.graph import StarLayerGraph, Namespace

EX = Namespace("http://example.org/")


## 1. A graph, a query, and the query as RDF

The examples below are grounded in this one data graph and one query.  

`sqe_parse_to_tree()` parses the query text into an RDF graph, using the `sqe:` ontology.  

In [2]:
from starlayer.sparql import sqe
from starlayer.sparql.vocab import SPEXPR

g = StarLayerGraph()
g.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:bob ex:knows ex:carol ;
        ex:likes ex:dana .
""", format="turtle")

print("data graph:")
print(g.serialize())


# a simple sparql query is converted to a sqe syntax tree
text_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?o WHERE { ?s ex:knows ?o }"""

# run the query against the data graph, as ordinary SPARQL text
print("query result:")
for row in g.query(text_query):
    print(g.qname(row.s), g.qname(row.o))
print()

# tree is a StarLayerGraph.
# root is the base node of the query graph.
tree, root = sqe.sqe_parse_to_tree(text_query)

tree.bind("sqe", sqe.SQE)
tree.bind("spexpr", SPEXPR)

print(tree.serialize())

data graph:
@prefix ex: <http://example.org/> .

ex:bob ex:knows ex:carol ;
    ex:likes ex:dana .

query result:
ex:bob ex:carol



@prefix ns1: <http://example.org/> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .

_:Nbd2d0e1c0b0943938de24b112eb31869 a sqe:Query, sqe:SelectQuery ;
    sqe:projection ( "s"^^spexpr:Variable "o"^^spexpr:Variable ) ;
    sqe:prologuePrefix [
        a sqe:PrefixBinding ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:where ( [
        a sqe:TriplePattern ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate ns1:knows ;
        sqe:subject "s"^^spexpr:Variable
    ] ) .



### 1.1 Giving the query a stable identity with `root=`

`sqe_parse_to_tree()` defaults to a fresh `BNode` for the query's own root node - as seen above. Passing `root=` instead gives it a real, caller-chosen identity, so it stays findable by that identity even once a graph accumulates more than one encoded query.

In [3]:
from rdflib import URIRef

query_iri = URIRef("http://example.org/queries#bobKnows")
named_tree, named_root = sqe.sqe_parse_to_tree(text_query, root=query_iri)

print("root is the given IRI:", named_root == query_iri)

named_tree.bind("sqe", sqe.SQE)
named_tree.bind("q", "http://example.org/queries#")
print(named_tree.serialize())

root is the given IRI: True
@prefix ns1: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix ns2: <http://example.org/> .
@prefix q: <http://example.org/queries#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .

q:bobKnows a sqe:Query, sqe:SelectQuery ;
    sqe:projection ( "s"^^ns1:Variable "o"^^ns1:Variable ) ;
    sqe:prologuePrefix [
        a sqe:PrefixBinding ;
        sqe:namespace <http://example.org/> ;
        sqe:prefixLabel "ex"^^ns1:PlainString
    ] ;
    sqe:where ( [
        a sqe:TriplePattern ;
        sqe:object "o"^^ns1:Variable ;
        sqe:predicate ns2:knows ;
        sqe:subject "s"^^ns1:Variable
    ] ) .



## 2. Edit the query by editing the RDF

The query graph can be edited as an RDF graph.


In [4]:
from rdflib import RDF, RDFS, Literal

# a triple term in the query is located and replaced.
for s, p, o in list(tree.triples((None, sqe.SQE.predicate, EX.knows))):
    tree.remove((s, p, o))
    tree.add((s, p, EX.likes))
    pattern_node = s

# a plain annotation on the query as a whole
tree.add((root, RDFS.label, Literal("bob's likes, not knows")))

# a real RDF 1.2 reification: the triple pattern's own rdf:type assertion
# is annotated to record why the predicate was changed
reifier = tree.add_reifier_annotation(RDFS.comment, Literal("predicate changed from ex:knows to ex:likes"))
tree.add_reification(reifier, (pattern_node, RDF.type, sqe.SQE.TriplePattern))

print(tree.serialize())

@version "1.2" .
@prefix ns1: <http://example.org/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix spexpr: <https://github.com/hidden-graph/starlayer/ns/sparql/expr#> .
@prefix sqe: <https://github.com/hidden-graph/starlayer/ns/sparql/query-edit#> .

_:Nbd2d0e1c0b0943938de24b112eb31869 a sqe:Query, sqe:SelectQuery ;
    rdfs:label "bob's likes, not knows" ;
    sqe:projection ( "s"^^spexpr:Variable "o"^^spexpr:Variable ) ;
    sqe:prologuePrefix [
        a sqe:PrefixBinding ;
        sqe:namespace ns1: ;
        sqe:prefixLabel "ex"^^spexpr:PlainString
    ] ;
    sqe:where ( [
        a sqe:TriplePattern {| rdfs:comment "predicate changed from ex:knows to ex:likes" |} ;
        sqe:object "o"^^spexpr:Variable ;
        sqe:predicate ns1:likes ;
        sqe:subject "s"^^spexpr:Variable
    ] ) .



## 3. Decode back into a real `Query`, and run it

`sqe_tree_to_query()` turns the edited RDF back into an executable `Query`. The edits to the query graph that are not relevant to the semantics of the query are ignored.

Running it against the *same* graph from section 1 proves the edit is semantic — the result set genuinely changes, from `(bob, carol)` to `(bob, dana)`.

In [5]:
# rdf graph becomes a query object
modified_query = sqe.sqe_tree_to_query(tree, root)

# query is executed
result = g.query(modified_query)

for row in result:
    print(g.qname(row.s), g.qname(row.o))

ex:bob ex:dana


## 4. Export the modified query back to text

`sqe_tree_to_text()` renders the edited RDF graph back to SPARQL text. Note that annotations aren't preserved in the text form - to keep them, store the query as a graph, not text.

In [6]:
print(sqe.sqe_tree_to_text(tree, root))

PREFIX ex: <http://example.org/>
SELECT ?s ?o
WHERE { ?s ex:likes ?o . }


## 5. Validate — automatically on parse, and again after editing outside the parse

`sqe_parse_to_tree()` validates the tree it produces against `sqe:`'s own SHACL shapes (`sqe:SelectQueryShape` among them). If the parsed tree doesn't conform, it raises `SQEValidationError` - see 5.1 below.

That check only happens at the moment of parsing. `sqe_validate()` can be called whenever a tree needs (re-)validating - for example after editing a tree by hand, since those edits happen outside of `sqe_parse_to_tree()` and aren't automatically re-checked.

In [7]:
# validate the edited syntax tree from above.
conforms, report_graph, report_text = sqe.sqe_validate(tree)
print("conforms:", conforms)

conforms: True


Editing the tree can also make it invalid. Removing a `sqe:TriplePattern`'s own required `sqe:predicate` is one example - `sqe_validate()` catches it:

In [8]:
# break the tree: remove the triple pattern's own required predicate
pattern_node = next(tree.subjects(RDF.type, sqe.SQE.TriplePattern))
tree.remove((pattern_node, sqe.SQE.predicate, None))

conforms, broken_report_graph, broken_report_text = sqe.sqe_validate(tree)
print("conforms:", conforms)
print(broken_report_text)

conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:maxCount Literal("1", datatype=xsd:integer) ; sh:message Literal("sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path") ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:node sqe:PredicateOrVariableOrPathShape ; sh:path sqe:predicate ]
	Focus Node: [ rdf:type rdfs:Resource, sqe:TriplePattern ; sqe:object Literal("o", datatype=spexpr:Variable) ; sqe:subject Literal("s", datatype=spexpr:Variable) ]
	Result Path: sqe:predicate
	Message: sqe:TriplePattern must have exactly one sqe:predicate, an IRI/Variable/property path



### 5.1 A query that's valid SPARQL, but fails SHACL validation

Valid SPARQL queries can still be wrong. For example `SELECT ?s ?z WHERE { ?s ex:knows ?o }` parses and runs fine as SPARQL. `?z` is never bound by the pattern, so it comes back unbound in every result row. SPARQL itself has no objection to this, but it almost always signals a typo or an incomplete edit: an "output" (`?z` projected) with no matching "input" (nothing in `?s ex:knows ?o` ever binds it).

`sqe:SelectQueryShape` carries a SHACL constraint able to catch exactly this.

Because `sqe_parse_to_tree()` validates automatically, parsing this query raises a `SQEValidationError` directly, with the full report attached as `.report_text`/`.report_graph`.

In [9]:
bad_query = """PREFIX ex: <http://example.org/>
SELECT ?s ?z WHERE { ?s ex:knows ?o }"""

# still valid SPARQL - runs fine, ?z just comes back unbound
for row in g.query(bad_query):
    print(g.qname(row.s), row.z)
print()

# sqe_parse_to_tree() raises SQEValidationError instead of returning a
# tree that's already known to fail sqe:'s own shapes
try:
    bad_tree, bad_root = sqe.sqe_parse_to_tree(bad_query)
except sqe.SQEValidationError as exc:
    print("conforms: False")
    print(exc.report_text)

ex:bob None



conforms: False
Validation Report
Conforms: False
Results (1):
Constraint Violation in SPARQLConstraintComponent (http://www.w3.org/ns/shacl#SPARQLConstraintComponent):
	Severity: sh:Violation
	Source Shape: sqe:SelectQueryShape
	Focus Node: [ rdf:type rdfs:Resource, sqe:Query, sqe:SelectQuery ; sqe:projection ( Literal("s", datatype=spexpr:Variable) Literal("z", datatype=spexpr:Variable) ) ; sqe:prologuePrefix [ rdf:type rdfs:Resource, sqe:PrefixBinding ; sqe:namespace <http://example.org/> ; sqe:prefixLabel Literal("ex", datatype=spexpr:PlainString) ] ; sqe:where ( [ rdf:type rdfs:Resource, sqe:TriplePattern ; sqe:object Literal("o", datatype=spexpr:Variable) ; sqe:predicate <http://example.org/knows> ; sqe:subject Literal("s", datatype=spexpr:Variable) ] ) ]
	Value Node: [ rdf:type rdfs:Resource, sqe:Query, sqe:SelectQuery ; sqe:projection ( Literal("s", datatype=spexpr:Variable) Literal("z", datatype=spexpr:Variable) ) ; sqe:prologuePrefix [ rdf:type rdfs:Resource, sqe:PrefixBindin